# Program-defined perturbation-class occupancy analysis

This notebook defines cell states using the **global dominant cNMF program**, not Leiden clusters. It then asks whether biological targets redistribute cells among the curated program-defined states.

The workflow does **not** use target-to-program CRT effects or target-to-program differential-effect results.

Primary state rule:

1. Compute the global argmax across all 50 cell-level cNMF usages.
2. Do not force cells into the curated set.
3. Retain a cell in the primary analysis only when its global dominant program is in the curated primary table and the program's expected day matches the observed collection day.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, zipfile
import pandas as pd
import yaml
from IPython.display import display, Image, SVG

NOTEBOOK_PATH = Path.cwd()
PACKAGE_DIR = NOTEBOOK_PATH.parent if NOTEBOOK_PATH.name == "notebooks" else NOTEBOOK_PATH
PACKAGE_DIR = PACKAGE_DIR.resolve()
CONFIG = PACKAGE_DIR / "config" / "program_defined_class_occupancy_config.yaml"

print("PACKAGE_DIR:", PACKAGE_DIR)
print("CONFIG:", CONFIG)
assert CONFIG.exists(), CONFIG


## 1. Inspect the curated program-state table

The primary list was transcribed from the first Hunter/Kyle annotation table. Programs flagged as uncertain or recommended for removal are retained only for sensitivity analysis, and programs marked for elimination are excluded.

In [ ]:
cfg = yaml.safe_load(CONFIG.read_text())
program_table_path = PACKAGE_DIR / cfg["curated_program_states_tsv"]
program_table = pd.read_csv(program_table_path, sep="\t")

display(program_table)
print("Primary programs:", program_table.query("use_in_primary == True")["program_id"].tolist())
print("Expanded sensitivity programs:", program_table.query("use_in_expanded_sensitivity == True")["program_id"].tolist())


## 2. Verify the real CC-Perturb-seq H5AD

The path below was recovered from the earlier analysis notebooks. The object should contain D0-D3 metadata, guide assignments, and cell-level cNMF usages.

In [ ]:
query_path = Path(cfg["query_h5ad"])
print(query_path)
assert query_path.exists(), f"Query H5AD does not exist: {query_path}"
print("Size (GB):", round(query_path.stat().st_size / 1e9, 2))


## 3. Export program-defined cell states and target-union assignments

This step reads the H5AD once, computes the global dominant program across all 50 programs, maps targeting guides to `intended_target_name`, and writes compact analysis tables.

In [ ]:
export_script = PACKAGE_DIR / "scripts" / "00_export_program_defined_occupancy_inputs.py"
cmd = [sys.executable, str(export_script), "--config", str(CONFIG)]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=PACKAGE_DIR)


## 4. Audit program-state coverage

Important checks:

- fraction of each observed day represented by the primary curated program states;
- replicate representation;
- non-curated dominant programs that are excluded from the primary model;
- carrier coverage by day.


In [ ]:
cfg = yaml.safe_load(CONFIG.read_text())
export_dir = Path(cfg["export_dir"])
if not export_dir.is_absolute():
    export_dir = PACKAGE_DIR / export_dir

for fn in [
    "program_state_assignment_qc.tsv",
    "program_state_counts_by_day_and_replicate.tsv",
    "dominant_program_counts_by_day.tsv",
    "control_definition_qc.tsv",
    "target_carrier_counts_by_day.tsv",
]:
    path = export_dir / fn
    print("\n###", fn)
    display(pd.read_csv(path, sep="\t").head(50))


## 5. Fit target-level occupancy models and create the class summary

For each observed day, target, and curated dominant-program state, the script fits a one-versus-rest logistic model with the established technical covariates. Target classes are summarized only after the target-level models are fitted.

In [ ]:
analysis_script = PACKAGE_DIR / "scripts" / "01_fit_program_defined_occupancy_and_plot.py"
cmd = [sys.executable, str(analysis_script), "--config", str(CONFIG)]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=PACKAGE_DIR)


## 6. Inspect the main results and coverage

In [ ]:
cfg = yaml.safe_load(CONFIG.read_text())
result_dir = Path(cfg["output_dir"])
if not result_dir.is_absolute():
    result_dir = PACKAGE_DIR / result_dir

for fn in [
    "class_program_state_occupancy_summary.tsv",
    "class_coverage_summary.tsv",
    "coverage_and_filtering.tsv",
    "target_class_program_state_raw_distribution.tsv",
    "analysis_audit.json",
]:
    path = result_dir / fn
    print("\n###", fn)
    if path.suffix == ".json":
        print(path.read_text())
    else:
        display(pd.read_csv(path, sep="\t").head(80))


## 7. Display the publication panel and supplementary heatmap

In [ ]:
main_png = result_dir / "Fig2_perturbation_class_program_state_occupancy.png"
supp_png = result_dir / "Fig2_target_program_state_occupancy_heatmap_supplementary.png"

display(Image(filename=str(main_png)))
display(Image(filename=str(supp_png)))


## 8. Package the completed outputs

In [ ]:
zip_path = PACKAGE_DIR / "program_defined_occupancy_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in result_dir.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(result_dir.parent)))
    for p in export_dir.glob("*.tsv"):
        zf.write(p, arcname=str(p.relative_to(export_dir.parent)))
    for p in export_dir.glob("*.json"):
        zf.write(p, arcname=str(p.relative_to(export_dir.parent)))
print("Wrote:", zip_path)
